In [1]:
# Davidson Bradley-Terry ratings for a full football league from Wyscout-style event JSONs
# Input folder:  /content/drive/MyDrive/Event data/Eredivisie/2024-2025/
# Output files: ratings CSV + match dataset CSV saved to same folder

from google.colab import drive
drive.mount("/content/drive")

import os, json, glob, math, re
import numpy as np
import pandas as pd
from scipy.optimize import minimize

DATA_DIR = "/content/drive/MyDrive/Event data/Eredivisie/2024-2025"
OUT_RATINGS = os.path.join(DATA_DIR, "bradley_terry_davidson_ratings.csv")
OUT_MATCHES = os.path.join(DATA_DIR, "bradley_terry_match_dataset.csv")

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def extract_match_from_events(path):
    data = load_json(path)
    events = data["events"] if isinstance(data, dict) and "events" in data else data

    teams = {}
    goals = {}
    xg = {}
    shots = {}

    for e in events:
        team = e.get("team") or {}
        opp = e.get("opponentTeam") or {}

        if team.get("id") is not None and team.get("name"):
            teams[team["id"]] = team["name"]
        if opp.get("id") is not None and opp.get("name"):
            teams[opp["id"]] = opp["name"]

        tid = team.get("id")
        if tid is None:
            continue

        goals.setdefault(tid, 0)
        xg.setdefault(tid, 0.0)
        shots.setdefault(tid, 0)

        shot = e.get("shot")
        if shot:
            shots[tid] += 1
            xg[tid] += float(shot.get("xg") or 0)

            secondary = e.get("type", {}).get("secondary", []) or []
            if shot.get("isGoal") or "goal" in secondary:
                goals[tid] += 1

    if len(teams) != 2:
        return None

    team_ids = list(teams.keys())
    home_id, away_id = team_ids[0], team_ids[1]

    filename = os.path.basename(path)
    m = re.match(r"(.+?) - (.+?),", filename)
    if m:
        home_name, away_name = m.group(1).strip(), m.group(2).strip()
        for tid, name in teams.items():
            if name == home_name:
                home_id = tid
            if name == away_name:
                away_id = tid

    hg = goals.get(home_id, 0)
    ag = goals.get(away_id, 0)

    if hg > ag:
        result = "H"
    elif ag > hg:
        result = "A"
    else:
        result = "D"

    return {
        "file": filename,
        "home_team": teams[home_id],
        "away_team": teams[away_id],
        "home_goals": hg,
        "away_goals": ag,
        "home_xg": xg.get(home_id, 0.0),
        "away_xg": xg.get(away_id, 0.0),
        "home_shots": shots.get(home_id, 0),
        "away_shots": shots.get(away_id, 0),
        "result": result
    }

paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.json")))
matches = []

for path in paths:
    row = extract_match_from_events(path)
    if row:
        matches.append(row)

df = pd.DataFrame(matches)

if df.empty:
    raise ValueError(f"No valid match JSON files found in: {DATA_DIR}")

teams = sorted(set(df["home_team"]) | set(df["away_team"]))
team_to_idx = {team: i for i, team in enumerate(teams)}
n = len(teams)

def neg_log_likelihood(params):
    # Last team rating fixed at 0 for identifiability
    ratings = np.r_[params[:n-1], 0.0]
    home_adv = params[n-1]
    draw_log = params[n]
    draw_strength = np.exp(draw_log)

    ll = 0.0

    for _, r in df.iterrows():
        hi = team_to_idx[r["home_team"]]
        ai = team_to_idx[r["away_team"]]

        home_power = np.exp(ratings[hi] + home_adv)
        away_power = np.exp(ratings[ai])
        draw_power = draw_strength * np.sqrt(home_power * away_power)

        denom = home_power + away_power + draw_power

        if r["result"] == "H":
            p = home_power / denom
        elif r["result"] == "A":
            p = away_power / denom
        else:
            p = draw_power / denom

        ll += np.log(max(p, 1e-12))

    # Small ridge penalty to prevent extreme ratings in small samples
    penalty = 0.01 * np.sum(ratings ** 2)
    return -ll + penalty

x0 = np.zeros(n + 1)  # n-1 ratings + home advantage + draw log
res = minimize(neg_log_likelihood, x0, method="BFGS")

ratings = np.r_[res.x[:n-1], 0.0]
home_adv = res.x[n-1]
draw_strength = np.exp(res.x[n])

ratings_df = pd.DataFrame({
    "team": teams,
    "bt_rating_log": ratings,
    "bt_rating_scaled": 1500 + 400 * ratings / math.log(10)
}).sort_values("bt_rating_log", ascending=False)

ratings_df["rank"] = range(1, len(ratings_df) + 1)
ratings_df = ratings_df[["rank", "team", "bt_rating_log", "bt_rating_scaled"]]

df.to_csv(OUT_MATCHES, index=False)
ratings_df.to_csv(OUT_RATINGS, index=False)

print(f"Loaded {len(df)} matches from {len(paths)} JSON files")
print(f"Teams: {len(teams)}")
print(f"Home advantage log-strength: {home_adv:.3f}")
print(f"Draw strength: {draw_strength:.3f}")
print(f"Saved ratings to: {OUT_RATINGS}")
print(f"Saved match dataset to: {OUT_MATCHES}")

display(ratings_df)

Mounted at /content/drive
Loaded 308 matches from 308 JSON files
Teams: 18
Home advantage log-strength: 0.571
Draw strength: 0.907
Saved ratings to: /content/drive/MyDrive/Event data/Eredivisie/2024-2025/bradley_terry_davidson_ratings.csv
Saved match dataset to: /content/drive/MyDrive/Event data/Eredivisie/2024-2025/bradley_terry_match_dataset.csv


,rank,team,bt_rating_log,bt_rating_scaled
1,1,Ajax,3.282566,2070.240047
12,2,PSV,3.150406,2047.281637
16,3,Utrecht,2.153771,1874.148266
3,4,Feyenoord,2.132955,1870.532223
0,5,AZ,2.120350,1868.342576
15,6,Twente,1.638279,1784.598234
5,7,Go Ahead Eagles,1.452475,1752.320666
10,8,NEC,1.020768,1677.325640
14,9,Sparta Rotterdam,0.984799,1671.077131
6,10,Groningen,0.891225,1654.821704
